In [13]:
import readabs_py
import pandas as pd
from MacroBackend import charting_plotly, Pull_Data

In [14]:
masterIndex = pd.read_hdf("/home/totabilcat/Documents/Code/Bootleg_Macro/MacroBackend/ABS_backend/abs_master_index.h5", key='data')
print(masterIndex.columns, len(masterIndex))

Index(['Data Item Description', 'Series Type', 'Series ID', 'Series Start',
       'Series End', 'No. Obs.', 'Unit', 'Data Type', 'Freq.',
       'Collection Month', 'Table', 'Table Description', 'Catalogue number'],
      dtype='object') 103094


In [15]:
csv_master = pd.read_csv("/home/totabilcat/Documents/Code/Bootleg_Macro/MacroBackend/ABS_backend/ABS_Series_MasterIndex.csv", index_col=0)

/tmp/ipykernel_150531/276903407.py:1: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  csv_master = pd.read_csv("/home/totabilcat/Documents/Code/Bootleg_Macro/MacroBackend/ABS_backend/ABS_Series_MasterIndex.csv", index_col=0)


In [16]:
masterIndex.loc["A130392586J"]

Data Item Description                   Index Numbers ;  Rents ;  Sydney ;
Series Type                                                       Original
Series ID                                                      A130392586J
Series Start                                           2022-07-01 00:00:00
Series End                                             2026-01-01 00:00:00
No. Obs.                                                                43
Unit                                                         Index Numbers
Data Type                                                            INDEX
Freq.                                                                Month
Collection Month                                                         1
Table                                                              6401010
Table Description        CPI: Group, Sub-group and Expenditure Class, I...
Catalogue number                                                    6401.0
Name: A130392586J, dtype:

In [17]:
print("len of master index:", len(masterIndex), f"\nColumns: {masterIndex.columns.tolist()}")
print("Length of csv master index:", len(csv_master), f"\nColumns: {csv_master.columns.tolist()}")

len of master index: 103094 
Columns: ['Data Item Description', 'Series Type', 'Series ID', 'Series Start', 'Series End', 'No. Obs.', 'Unit', 'Data Type', 'Freq.', 'Collection Month', 'Table', 'Table Description', 'Catalogue number']
Length of csv master index: 195203 
Columns: ['Data Item Description', 'Series Type', 'Series ID.1', 'Series Start', 'Series End', 'No. Obs.', 'Unit', 'Data Type', 'Freq.', 'Collection Month', 'Table', 'Table Description', 'Catalogue number']


In [18]:
# # Rebuild ABS master index by concatenating each catalogue's series index (axis=0)
# catalogues = catalog.index.astype(str)

# cat_indexes = {}
# failed_catalogues = []
# no_index_catalogues = {}

# for cat_num in catalogues:
#     try:
#         # read_abs_cat returns a tuple; [1] is the index DataFrame
#         cat_data = readabs.read_abs_cat(cat_num)
#         if isinstance(cat_data, tuple) and len(cat_data) > 1:
#             cat_index = cat_data[1]
#         else:
#             no_index_catalogues[cat_num] = cat_data
#             continue

#         if cat_index is None or len(cat_index) == 0:
#             failed_catalogues.append((cat_num, "empty index returned"))
#             continue

#         cat_indexes[cat_num] = cat_index.copy()
#     except Exception as exc:
#         failed_catalogues.append((cat_num, str(exc)))

# masterIndex = (
#     pd.concat(cat_indexes, axis=0)
#     .reset_index(level=0)
#     .rename(columns={"level_0": "Catalogue number"})
#     if cat_indexes
#     else pd.DataFrame()
#  )

# print(f"Built masterIndex from {len(cat_indexes)} catalogues")
# print(f"masterIndex rows: {len(masterIndex)}")
# print(f"Failed catalogues: {len(failed_catalogues)}")
# if failed_catalogues:
#     failed_df = pd.DataFrame(failed_catalogues, columns=["Catalogue number", "Error"])
#     display(failed_df.head(20))

In [19]:
# 1) Find rows where catalogue is missing
mask = masterIndex["Catalogue number"].isna()

# 2) Extract leading number from "Table" (e.g. "8501.0 Retail Trade, Australia" -> "8501.0")
extracted = masterIndex.loc[mask, "Table"].str.extract(r"^\s*([0-9]+(?:\.[0-9]+)?)")[0]

# 3) Fill only missing catalogue numbers
masterIndex.loc[mask, "Catalogue number"] = pd.to_numeric(extracted, errors="coerce")

In [20]:
import abs_series_by_r as rabs

In [21]:
loc = rabs.abs_download_with_r(series_id="A84423050A")

[R] Downloaded series A84423050A to: /home/totabilcat/Documents/Code/Bootleg_Macro/User_Data/ABS/LastPull/6202001.xlsx
[Excel] Reading /home/totabilcat/Documents/Code/Bootleg_Macro/User_Data/ABS/LastPull/6202001.xlsx
[Excel] Looking for series: A84423050A
[Excel] Found A84423050A in column 64 (name: 'Unemployment rate ;  Persons ;.1')
[File] Copied to Full_Sheets: /home/totabilcat/Documents/Code/Bootleg_Macro/User_Data/ABS/Full_Sheets/6202001.xlsx
[File] Deleted old file: 634501.xlsx
[Excel] Created series 'Unemployment rate ;  Persons ;1' with 577 observations


In [22]:
loc

(1978-02-01    6.644543
 1978-03-01    6.302724
 1978-04-01    6.267395
 1978-05-01    6.209378
 1978-06-01    6.303954
                 ...   
 2025-10-01    4.333151
 2025-11-01    4.317995
 2025-12-01    4.113862
 2026-01-01    4.073268
 2026-02-01    4.277978
 Name: Unemployment rate ;  Persons ;1, Length: 577, dtype: float64,
 Unit                                        Percent
 Series Type                     Seasonally Adjusted
 Data Type                                   PERCENT
 Frequency                                     Month
 Collection Month                                  1
 Series Start                    1978-02-01 00:00:00
 Series End                      2026-02-01 00:00:00
 No. Obs                                         577
 Series ID                                A84423050A
 title               Unemployment rate ;  Persons ;1
 Name: A84423050A, dtype: object)

In [30]:
rents = pd.read_excel("/media/totabilcat/SHARED/Shared_Docs/Studies/Aus/ABS_Data/640107_q32024.xlsx", sheet_name="Data1", index_col=0)
rents["Index Numbers ;  Rents ;  Sydney ;"].head(20)

Unit                         Index Numbers
Series Type                       Original
Data Type                            INDEX
Frequency                          Quarter
Collection Month                         3
Series Start           1972-09-01 00:00:00
Series End             2025-09-01 00:00:00
No. Obs                                213
Series ID                        A2331836L
1948-09-01 00:00:00                    NaN
1948-12-01 00:00:00                    NaN
1949-03-01 00:00:00                    NaN
1949-06-01 00:00:00                    NaN
1949-09-01 00:00:00                    NaN
1949-12-01 00:00:00                    NaN
1950-03-01 00:00:00                    NaN
1950-06-01 00:00:00                    NaN
1950-09-01 00:00:00                    NaN
1950-12-01 00:00:00                    NaN
1951-03-01 00:00:00                    NaN
Name: Index Numbers ;  Rents ;  Sydney ;, dtype: object

In [44]:
from MacroBackend import charting_plotly

In [46]:
rents_meta = rents["Index Numbers ;  Rents ;  Sydney ;"].iloc[0:9]
rents_data = rents["Index Numbers ;  Rents ;  Sydney ;"].iloc[9:]
rents_data.index = pd.to_datetime(rents_data.index)
rents_data = rents_data.astype(float).dropna()

In [40]:
latterRents = pd.read_excel("/media/totabilcat/SHARED/Shared_Docs/Studies/Aus/ABS_Data/6401010.xlsx", sheet_name="Data1", index_col=0)

In [47]:
later_metadata = latterRents["Index Numbers ;  Rents ;  Sydney ;"].iloc[0:9]
later_data = latterRents["Index Numbers ;  Rents ;  Sydney ;"].iloc[9:]
later_data.index = pd.to_datetime(later_data.index)
later_data = later_data.astype(float).dropna()

In [52]:
plot = charting_plotly.dual_axis_basic_plot({rents_data.name: rents_data}, secondary_data={later_data.name: later_data}, 
                                            title="Sydney Rental price Index from CPI", 
                                            primary_yaxis_title="Index Number")
y_min = min(rents_data.min(), later_data.min())
y_max = max(rents_data.max(), later_data.max())
plot.update_layout(
    yaxis=dict(range=[y_min, y_max]),
    yaxis2=dict(range=[y_min, y_max])
)

plot.show()


In [53]:
later_data.tail(5)

2025-09-01    100.00
2025-10-01    100.20
2025-11-01    100.59
2025-12-01    100.84
2026-01-01    101.15
Name: Index Numbers ;  Rents ;  Sydney ;, dtype: float64

In [56]:
rents_rebased = (rents_data/rents_data.iloc[-1]) * 100
rents_rebased.tail(5)

2024-09-01     96.845878
2024-12-01     97.491039
2025-03-01     98.351254
2025-06-01     99.211470
2025-09-01    100.000000
Name: Index Numbers ;  Rents ;  Sydney ;, dtype: float64

In [76]:
full_series = pd.concat([rents_rebased[0:-2].resample('MS').mean(), later_data], axis=0)
full_series.head(20)

1972-09-01    6.093190
1972-10-01         NaN
1972-11-01         NaN
1972-12-01    6.164875
1973-01-01         NaN
1973-02-01         NaN
1973-03-01    6.236559
1973-04-01         NaN
1973-05-01         NaN
1973-06-01    6.379928
1973-07-01         NaN
1973-08-01         NaN
1973-09-01    6.523297
1973-10-01         NaN
1973-11-01         NaN
1973-12-01    6.666667
1974-01-01         NaN
1974-02-01         NaN
1974-03-01    6.738351
1974-04-01         NaN
Name: Index Numbers ;  Rents ;  Sydney ;, dtype: float64

In [77]:
full_series_clean = full_series.dropna().sort_index()
full_series_clean = full_series_clean[~full_series_clean.index.duplicated(keep='first')]
full_series_clean

1972-09-01      6.093190
1972-12-01      6.164875
1973-03-01      6.236559
1973-06-01      6.379928
1973-09-01      6.523297
                 ...    
2025-09-01    100.000000
2025-10-01    100.200000
2025-11-01    100.590000
2025-12-01    100.840000
2026-01-01    101.150000
Name: Index Numbers ;  Rents ;  Sydney ;, Length: 243, dtype: float64

In [78]:
bang = charting_plotly.dual_axis_basic_plot({rents_rebased.name: rents_rebased}, secondary_data={full_series.name: full_series_clean}, 
                                            title="Sydney Rental price Index from CPI", 
                                            primary_yaxis_title="Index Number")
y_min = full_series.min()
y_max = full_series.max()
bang.update_layout(
    yaxis=dict(range=[y_min, y_max]),
    yaxis2=dict(range=[y_min, y_max])
)

bang.show()

In [80]:
full_series_clean.index[0]

Timestamp('1972-09-01 00:00:00')

In [82]:
later_metadata.loc["Series Start"] = full_series_clean.index[0]
later_metadata

Unit                      Index Numbers
Series Type                    Original
Data Type                         INDEX
Frequency                         Month
Collection Month                      1
Series Start        1972-09-01 00:00:00
Series End          2026-01-01 00:00:00
No. Obs                              43
Series ID                   A130392586J
Name: Index Numbers ;  Rents ;  Sydney ;, dtype: object

In [83]:
abs_db = pd.HDFStore("/home/totabilcat/Documents/Code/Bootleg_Macro/MacroBackend/ABS_backend/abs_db_custom.h5")
abs_db["Sydney_Rents_CPI"] = full_series_clean
abs_db["Sydney_Rents_CPI_metadata"] = later_metadata
abs_db.close()



/tmp/ipykernel_150531/3689601449.py:3: PerformanceWarning:


your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->values] [items->None]


